# Insert Email Body into Dataverse

Inserts a single record into the Dataverse table **`ubscee_tablename`** containing:
- **EmailBody**  – the dynamic HTML email body (workspace-monitoring notification)
- **EmailSubject** – `[FUAM] email test`

Uses the [Dataverse Web API](https://learn.microsoft.com/power-apps/developer/data-platform/webapi/create-entity-web-api): a `POST` to the entity set returns `HTTP 204 No Content` with the new record URI in the `OData-EntityId` header.

In [ ]:
import requests
import notebookutils

In [ ]:
# --- Configuration ---
dataverse_env_url = "https://org0f651234be2.crm4.dynamics.com"

tenant_id  = "9e929790-272d-4977-a2ab-301443c11ece"
client_id  = "b5c04c9c-0588-418f-8f60-2d83d38cb635"

azure_key_vault_name = "ab-fabric-automation-akv"
secret_name          = "fabric-monit-secret"

api_version = "v9.2"

# Entity set name (plural) of the target table. For a custom table the entity
# set is usually the schema name pluralized, e.g. "ubscee_tablenames".
# Adjust if your environment differs.
entity_set_name = "ubscee_tablenames"

# Logical column names in Dataverse. Custom columns normally carry the
# publisher prefix (e.g. ubscee_emailbody). Update these to match your table.
col_email_body    = "ubscee_emailbody"
col_email_subject = "ubscee_emailsubject"

email_subject = "[FUAM] email test"

In [ ]:
# --- Email body HTML to insert ---
email_body = """<div style="font-family:Arial, sans-serif; font-size:14px; line-height:1.6;">
  <p style="font-style:italic; color:#666;">This notification is sent as part of Fabric Workspace Monitoring. <a href="https://myhub.ubs.net/myhub/assist/articles/420960">Learn more</a></p>
  <p>Dear Workspace Owner,</p>
  <p>Fabric Workspace Monitoring detected following artifacts and is in process of deletion, as they were not allowed in your workspace <strong>11111111-aaaa-bbbb-cccc-222222222222</strong>. As mentioned in <a href="https://myhub.ubs.net/myhub/assist/articles/420960">article link</a>.</p>
  <table border="1" style="border-collapse:collapse; font-family:Arial; width:100%; margin:20px 0;">
    <thead><tr style="background-color:#E60000; color:white; font-weight:bold;">
      <th style="padding:8px;">Workspace ID</th>
      <th style="padding:8px;">Item ID</th>
      <th style="padding:8px;">Item Type</th>
      <th style="padding:8px;">Creator ID</th>
      <th style="padding:8px;">Created At</th>
    </tr></thead><tbody>
      <tr>
        <td style="padding:8px;">11111111-aaaa-bbbb-cccc-222222222222</td>
        <td style="padding:8px;">33333333-dddd-eeee-ffff-444444444444</td>
        <td style="padding:8px;">Lakehouse</td>
        <td style="padding:8px;">john.doe@ubs.com</td>
        <td style="padding:8px;">2026-06-25T08:14:00Z</td>
      </tr>
      <tr>
        <td style="padding:8px;">11111111-aaaa-bbbb-cccc-222222222222</td>
        <td style="padding:8px;">55555555-6666-7777-8888-999999999999</td>
        <td style="padding:8px;">Notebook</td>
        <td style="padding:8px;">jane.smith@ubs.com</td>
        <td style="padding:8px;">2026-06-25T09:02:00Z</td>
      </tr>
    </tbody></table>
  <p>To learn more about Fabric, visit <a href="https://goto/powerhub">goto/powerhub</a>.</p>
  <p>For more information about constraints and limitations, visit <a href="#">Power Platform constraints</a> or <a href="#">Power BI constraints</a>.</p>
  <p><strong>Best regards,</strong><br>
  UBS BizApps Team<br>
  Group Functions</p>
  <hr style="border:none; border-top:1px solid #ccc; margin:20px 0;">
  <p style="font-size:12px; color:#999;">
  Please do not reply to this automatically generated email as the mailbox is not monitored.<br>
  &copy; UBS 2026. For internal use only.
  </p>
</div>"""

print(f"Email body length: {len(email_body)} characters")

In [ ]:
# --- Get the service principal secret from Key Vault ---
azure_key_vault_url = f"https://{azure_key_vault_name}.vault.azure.net"
client_secret = notebookutils.credentials.getSecret(azure_key_vault_url, secret_name)

In [ ]:
# --- Acquire a Dataverse access token (client credentials / SPN) ---
def get_dataverse_token():
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "scope":         dataverse_env_url.rstrip("/") + "/.default",
    }
    response = requests.post(url, data=payload)
    response.raise_for_status()
    return response.json()["access_token"]

dataverse_token = get_dataverse_token()
print("Token acquired.")

In [ ]:
# --- Insert a single record via POST ---
def insert_record(record):
    url = f"{dataverse_env_url.rstrip('/')}/api/data/{api_version}/{entity_set_name}"
    headers = {
        "Authorization":    f"Bearer {dataverse_token}",
        "OData-MaxVersion": "4.0",
        "OData-Version":    "4.0",
        "Accept":           "application/json",
        "Content-Type":     "application/json",
    }
    response = requests.post(url, headers=headers, json=record)
    response.raise_for_status()
    # On success Dataverse returns 204 with the new record URI in this header
    return response.headers.get("OData-EntityId")

In [ ]:
# --- Insert the email record ---
record = {
    col_email_body:    email_body,
    col_email_subject: email_subject,
}

entity_id = insert_record(record)
print(f"Created: {entity_id}")